# AccessAI - Prototipo de detección urbana

Este cuaderno implementa un experimento reproducible sobre el ROD-Dataset:

1. Comprobar entorno y GPU.
2. Comprobar Ultralytics.
3. Preparar el ROD-Dataset.
4. Filtrar y remapear las 25 clases originales a 4 categorías.
5. Crear `data_filtrado.yaml`.
6. Entrenar YOLO26M con AutoBatch en la GPU disponible.
7. Cargar `best.pt`.
8. Evaluar el modelo en `test`.
9. Visualizar las curvas del entrenamiento.
10. Ejecutar inferencia sobre una imagen y guardar el resultado.

## Clases finales

| ID | Clase |
|---:|---|
| 0 | `Obstaculo_Dinamico` |
| 1 | `Obstaculo_Fijo` |
| 2 | `Barrera_Arquitectonica` |
| 3 | `Infraestructura_Peatonal` |

> Las etiquetas originales se conservan en `labels_originales`. El remapeo siempre usa esas etiquetas como fuente, por lo que ejecutar el notebook de nuevo no aplica un segundo remapeo.

> **Alcance:** estas cuatro categorías son una agrupación experimental del ROD-Dataset. No equivalen a `sidewalk` y `curbramp` y sus métricas no constituyen una validación de accesibilidad urbana específica.

## Nota de configuración

Se normaliza el notebook a **YOLO26M** porque es el modelo que aparece en el código actual y es mucho más razonable para una RTX 4060 Laptop GPU de 8 GB que intentar fijar un batch 64 con YOLO26X.

`batch=-1` activa AutoBatch de Ultralytics. No se debe describir `batch=64` como AutoBatch.

Para un experimento específico con `yolo26x.pt`, cambia `MODEL_NAME`, pero en 8 GB de VRAM conviene tratarlo como experimento separado y reducir agresivamente el batch.

## 01. Entorno y GPU

In [ ]:
from pathlib import Path
import sys
import torch

print("=" * 70)
print("ACCESSAI - ENTORNO Y GPU")
print("=" * 70)

print(f"Python: {sys.executable}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA de PyTorch: {torch.version.cuda}")
print(f"HIP: {torch.version.hip}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible en este kernel de Jupyter. "
        "Selecciona el entorno 'proyecto_yolo' con PyTorch CUDA."
    )

device = 0
gpu_name = torch.cuda.get_device_name(device)
props = torch.cuda.get_device_properties(device)

print(f"GPU: {gpu_name}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"VRAM total: {props.total_memory / 1024**3:.2f} GB")
print(f"Compute capability: {props.major}.{props.minor}")
print(
    f"VRAM asignada: "
    f"{torch.cuda.memory_allocated(device) / 1024**3:.2f} GB"
)
print(
    f"VRAM reservada: "
    f"{torch.cuda.memory_reserved(device) / 1024**3:.2f} GB"
)

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Prueba real de CUDA
x = torch.zeros((1, 3, 640, 640), device=device)
print("✅ Tensor CUDA de prueba creado")
del x
torch.cuda.empty_cache()
print("✅ GPU funcionando correctamente")


ACCESSAI - ENTORNO Y GPU
Python: c:\Users\Usuario\.conda\envs\proyecto_yolo\python.exe
PyTorch: 2.14.0+cu126
CUDA de PyTorch: 12.6
HIP: None
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU count: 1
VRAM total: 8.00 GB
Compute capability: 8.9
VRAM asignada: 0.00 GB
VRAM reservada: 0.00 GB
✅ Tensor CUDA de prueba creado
✅ GPU funcionando correctamente


## 02. Comprobar Ultralytics

In [12]:
import ultralytics

print("=" * 70)
print("ACCESSAI - ULTRALYTICS")
print("=" * 70)
print("Ultralytics:", ultralytics.__version__)


ACCESSAI - ULTRALYTICS
Ultralytics: 8.4.69


## 03. Configuración del proyecto y dataset

Se busca automáticamente una carpeta que contenga `DATA/ROD-Dataset/dataset`, evitando depender de una única carpeta de trabajo de Jupyter.

In [18]:
from pathlib import Path
import yaml
import shutil
from collections import Counter

def encontrar_raiz_proyecto():
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]

    if "__file__" in globals():
        try:
            candidatos.insert(0, Path(__file__).resolve().parent)
        except NameError:
            pass

    vistos = set()
    for candidato in candidatos:
        candidato = candidato.resolve()
        if candidato in vistos:
            continue
        vistos.add(candidato)

        dataset_candidates = [
            candidato / "DATA" / "ROD-Dataset" / "dataset",
            candidato / "DATA" / "dataset",
            candidato / "ROD-Dataset" / "dataset",
        ]

        if any(path.exists() for path in dataset_candidates):
            return candidato

        repo_markers = [
            candidato / "README.md",
            candidato / "PLAN.md",
            candidato / ".git",
        ]
        if any(marker.exists() for marker in repo_markers):
            data_dir = candidato / "DATA"
            if data_dir.exists() and any(
                (data_dir / "ROD-Dataset" / "dataset").exists(),
                (data_dir / "dataset").exists(),
            ):
                return candidato

    return actual

ROOT = encontrar_raiz_proyecto()
DATASET_CANDIDATES = [
    ROOT / "DATA" / "ROD-Dataset" / "dataset",
    ROOT / "DATA" / "dataset",
    ROOT / "ROD-Dataset" / "dataset",
]
DATASET_PATH = next(
    (path for path in DATASET_CANDIDATES if path.exists()),
    DATASET_CANDIDATES[0],
)

yaml_original_path = DATASET_PATH / "data.yaml"
yaml_nuevo_path = DATASET_PATH / "data_filtrado.yaml"

print("ROOT:")
print(ROOT)
print("\nDataset:")
print(DATASET_PATH)
print("\nExiste dataset:", DATASET_PATH.exists())
print("Existe data.yaml:", yaml_original_path.exists())

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        "No existe el dataset en ninguna de estas rutas:\n"
        + "\n".join(str(path) for path in DATASET_CANDIDATES)
    )

if not yaml_original_path.exists():
    raise FileNotFoundError(f"No existe:\n{yaml_original_path}")


ROOT:
C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2

Dataset:
C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset

Existe dataset: True
Existe data.yaml: True


## 04. Mapeo de clases del ROD-Dataset

In [14]:
CLASES_ORIGINALES = [
    "Bike",
    "Building",
    "Car",
    "Person",
    "Stairs",
    "Traffic sign",
    "Electrical Pole",
    "Road",
    "Motorcycle",
    "Dustbin",
    "Dog",
    "Manhole",
    "Tree",
    "Guard rail",
    "Pedestrian crosswalk",
    "Truck",
    "Bus",
    "Bench",
    "Traffic Cone",
    "Fire hydrant",
    "Teraffic Barrel",
    "Plant Pot",
    "Electrical Box",
    "Chair",
    "Bicycle Rack",
]

MAPEO_CLASES = {
    # 0 -> Obstaculo_Dinamico
    0: 0,   # Bike
    2: 0,   # Car
    3: 0,   # Person
    8: 0,   # Motorcycle
    10: 0,  # Dog
    15: 0,  # Truck
    16: 0,  # Bus

    # 1 -> Obstaculo_Fijo
    5: 1,   # Traffic sign
    6: 1,   # Electrical Pole
    9: 1,   # Dustbin
    12: 1,  # Tree
    13: 1,  # Guard rail
    17: 1,  # Bench
    18: 1,  # Traffic Cone
    19: 1,  # Fire hydrant
    20: 1,  # Teraffic Barrel
    21: 1,  # Plant Pot
    22: 1,  # Electrical Box
    23: 1,  # Chair
    24: 1,  # Bicycle Rack

    # 2 -> Barrera_Arquitectonica
    4: 2,   # Stairs

    # 3 -> Infraestructura_Peatonal
    7: 3,   # Road
    11: 3,  # Manhole
    14: 3,  # Pedestrian crosswalk
}

NUEVOS_NOMBRES = [
    "Obstaculo_Dinamico",
    "Obstaculo_Fijo",
    "Barrera_Arquitectonica",
    "Infraestructura_Peatonal",
]

CLASES_IGNORADAS = {
    1: "Building",
}

NUM_CLASSES = len(NUEVOS_NOMBRES)

# Comprobaciones estructurales
assert len(CLASES_ORIGINALES) == 25
assert len(MAPEO_CLASES) + len(CLASES_IGNORADAS) == len(CLASES_ORIGINALES)
assert set(MAPEO_CLASES) | set(CLASES_IGNORADAS) == set(range(25))
assert all(0 <= destino < NUM_CLASSES for destino in MAPEO_CLASES.values())

print("=" * 70)
print("ACCESSAI - MAPEO DE CLASES")
print("=" * 70)

print("\nClases finales:")
for idx, nombre in enumerate(NUEVOS_NOMBRES):
    print(f"{idx}: {nombre}")

print("\nMAPEO ORIGINAL -> FINAL")
for clase_original, clase_final in sorted(MAPEO_CLASES.items()):
    print(
        f"{clase_original:2d} | "
        f"{CLASES_ORIGINALES[clase_original]:25s} -> "
        f"{clase_final} | "
        f"{NUEVOS_NOMBRES[clase_final]}"
    )

print("\nCLASES IGNORADAS")
for clase, nombre in CLASES_IGNORADAS.items():
    print(f"{clase}: {nombre}")


ACCESSAI - MAPEO DE CLASES

Clases finales:
0: Obstaculo_Dinamico
1: Obstaculo_Fijo
2: Barrera_Arquitectonica
3: Infraestructura_Peatonal

MAPEO ORIGINAL -> FINAL
 0 | Bike                      -> 0 | Obstaculo_Dinamico
 2 | Car                       -> 0 | Obstaculo_Dinamico
 3 | Person                    -> 0 | Obstaculo_Dinamico
 4 | Stairs                    -> 2 | Barrera_Arquitectonica
 5 | Traffic sign              -> 1 | Obstaculo_Fijo
 6 | Electrical Pole           -> 1 | Obstaculo_Fijo
 7 | Road                      -> 3 | Infraestructura_Peatonal
 8 | Motorcycle                -> 0 | Obstaculo_Dinamico
 9 | Dustbin                   -> 1 | Obstaculo_Fijo
10 | Dog                       -> 0 | Obstaculo_Dinamico
11 | Manhole                   -> 3 | Infraestructura_Peatonal
12 | Tree                      -> 1 | Obstaculo_Fijo
13 | Guard rail                -> 1 | Obstaculo_Fijo
14 | Pedestrian crosswalk      -> 3 | Infraestructura_Peatonal
15 | Truck                     -> 0 |

## 05. Crear `data_filtrado.yaml`

In [15]:
with open(yaml_original_path, "r", encoding="utf-8") as f:
    config_original = yaml.safe_load(f)

print("Contenido original:")
print(config_original)

config_nueva = {
    "path": str(DATASET_PATH.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": NUM_CLASSES,
    "names": NUEVOS_NOMBRES,
}

with open(yaml_nuevo_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        config_nueva,
        f,
        sort_keys=False,
        allow_unicode=True,
    )

print("\n✅ Nuevo YAML creado:")
print(yaml_nuevo_path)

print("\nContenido:")
print(yaml_nuevo_path.read_text(encoding="utf-8"))


Contenido original:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 25, 'names': ['Bike', 'Building', 'Car', 'Person', 'Stairs', 'Traffic sign', 'Electrical Pole', 'Road', 'Motorcycle', 'Dustbin', 'Dog', 'Manhole', 'Tree', 'Guard rail', 'Pedestrian crosswalk', 'Truck', 'Bus', 'Bench', 'Traffic Cone', 'Fire hydrant', 'Teraffic Barrel', 'Plant Pot', 'Electrical Box', 'Chair', 'Bicycle Rack']}

✅ Nuevo YAML creado:
C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset\data_filtrado.yaml

Contenido:
path: C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA
  SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset
train: train/images
val: valid/images
test: test/images
nc: 4
names:
- Obstaculo_Dinamico
- Obstaculo_Fijo
- Barrera_Arquitectonica
- Infraestructura_Peatonal



## 06. Preparar etiquetas

La transformación siempre parte de `labels_originales`.

- Si `labels_originales` no existe, se conserva la carpeta `labels` actual.
- Se genera `labels_filtradas` desde las originales.
- Después se reemplaza `labels` por las etiquetas remapeadas.
- Al volver a ejecutar, nunca se utiliza el remapeo anterior como fuente.

In [16]:

from pathlib import Path
import shutil
from collections import Counter

def preparar_etiquetas(split):
    split_dir = DATASET_PATH / split
    labels_dir = split_dir / "labels"
    labels_originales_dir = split_dir / "labels_originales"
    labels_filtradas_dir = split_dir / "labels_filtradas"

    if not split_dir.exists():
        raise FileNotFoundError(f"No existe el split:\n{split_dir}")

    # 1) Siempre partir desde labels_originales
    if labels_originales_dir.exists():
        source_dir = labels_originales_dir
        print(f"↪ {split}: usando labels_originales como fuente")

    elif labels_dir.exists():
        print(f"↪ {split}: labels_originales no existe; se conserva labels actual")
        shutil.copytree(labels_dir, labels_originales_dir)
        source_dir = labels_originales_dir

    else:
        raise FileNotFoundError(f"No existe labels ni labels_originales en:\n{split_dir}")

    # 2) Preparar directorio temporal de salida
    if labels_filtradas_dir.exists():
        shutil.rmtree(labels_filtradas_dir)
    labels_filtradas_dir.mkdir(parents=True, exist_ok=True)

    # 3) Eliminar labels previo para evitar reusar remapeos antiguos
    if labels_dir.exists():
        shutil.rmtree(labels_dir)
    labels_dir.mkdir(parents=True, exist_ok=True)

    clases_originales = Counter()
    clases_nuevas = Counter()
    clases_descartadas = Counter()
    archivos_procesados = 0
    lineas_validas = 0
    lineas_invalidas = 0

    for archivo in sorted(source_dir.glob("*.txt")):
        archivos_procesados += 1
        ruta_salida = labels_filtradas_dir / archivo.name
        nuevas_lineas = []

        with open(archivo, "r", encoding="utf-8") as f:
            for num_linea, linea in enumerate(f, start=1):
                partes = linea.strip().split()
                if not partes:
                    continue

                try:
                    clase_original = int(partes[0])
                except ValueError:
                    lineas_invalidas += 1
                    print(f"⚠️ Clase inválida en {archivo.name}:{num_linea} -> {partes[0]}")
                    continue

                coordenadas = partes[1:]
                if len(coordenadas) < 6:
                    lineas_invalidas += 1
                    print(f"⚠️ Polígono demasiado pequeño en {archivo.name}:{num_linea}")
                    continue

                if len(coordenadas) % 2 != 0:
                    lineas_invalidas += 1
                    print(f"⚠️ Coordenadas impares en {archivo.name}:{num_linea}")
                    continue

                try:
                    valores = [float(x) for x in coordenadas]
                except ValueError:
                    lineas_invalidas += 1
                    print(f"⚠️ Coordenadas no numéricas en {archivo.name}:{num_linea}")
                    continue

                if any(v < 0 or v > 1 for v in valores):
                    lineas_invalidas += 1
                    print(f"⚠️ Coordenadas fuera de [0,1] en {archivo.name}:{num_linea}")
                    continue

                clases_originales[clase_original] += 1
                lineas_validas += 1

                if clase_original not in MAPEO_CLASES:
                    clases_descartadas[clase_original] += 1
                    continue

                clase_nueva = MAPEO_CLASES[clase_original]
                nueva_linea = [str(clase_nueva)] + partes[1:]
                nuevas_lineas.append(" ".join(nueva_linea) + "\n")
                clases_nuevas[clase_nueva] += 1

        ruta_salida.write_text("".join(nuevas_lineas), encoding="utf-8")

    # 4) Mover resultados finales a labels
    for archivo in sorted(labels_filtradas_dir.glob("*.txt")):
        destino = labels_dir / archivo.name
        shutil.move(str(archivo), str(destino))

    print("\n" + "-" * 70)
    print(f"Split: {split}")
    print("-" * 70)
    print(f"Archivos procesados: {archivos_procesados}")
    print(f"Líneas YOLO válidas: {lineas_validas}")
    print(f"Líneas inválidas: {lineas_invalidas}")

    print("\nClases finales:")
    for clase, nombre in enumerate(NUEVOS_NOMBRES):
        print(f"{clase} - {nombre}: {clases_nuevas[clase]}")

    print("\nClases descartadas:")
    if clases_descartadas:
        for clase, cantidad in sorted(clases_descartadas.items()):
            nombre = CLASES_ORIGINALES[clase] if 0 <= clase < 25 else "desconocida"
            print(f"{clase} - {nombre}: {cantidad}")
    else:
        print("Ninguna")

    print(f"\n✅ {split} preparado correctamente.")

for split in ["train", "valid", "test"]:
    preparar_etiquetas(split)

↪ train: usando labels_originales como fuente
⚠️ Polígono demasiado pequeño en IMG_00001.txt:1
⚠️ Polígono demasiado pequeño en IMG_00002.txt:1
⚠️ Polígono demasiado pequeño en IMG_00003.txt:1
⚠️ Polígono demasiado pequeño en IMG_00003.txt:2
⚠️ Polígono demasiado pequeño en IMG_00004.txt:1
⚠️ Polígono demasiado pequeño en IMG_00005.txt:1
⚠️ Polígono demasiado pequeño en IMG_00006.txt:1
⚠️ Polígono demasiado pequeño en IMG_00007.txt:1
⚠️ Polígono demasiado pequeño en IMG_00008.txt:1
⚠️ Polígono demasiado pequeño en IMG_00011.txt:1
⚠️ Polígono demasiado pequeño en IMG_00012.txt:1
⚠️ Polígono demasiado pequeño en IMG_00013.txt:1
⚠️ Polígono demasiado pequeño en IMG_00014.txt:1
⚠️ Polígono demasiado pequeño en IMG_00015.txt:1
⚠️ Polígono demasiado pequeño en IMG_00016.txt:1
⚠️ Polígono demasiado pequeño en IMG_00017.txt:1
⚠️ Polígono demasiado pequeño en IMG_00018.txt:1
⚠️ Polígono demasiado pequeño en IMG_00019.txt:1
⚠️ Polígono demasiado pequeño en IMG_00020.txt:1
⚠️ Polígono demasiado p

## 07. Comprobar estructura del dataset

In [19]:
print("=" * 70)
print("ACCESSAI - ESTRUCTURA DEL DATASET")
print("=" * 70)

for split in ["train", "valid", "test"]:
    split_dir = DATASET_PATH / split
    images_dir = split_dir / "images"
    labels_dir = split_dir / "labels"
    originals_dir = split_dir / "labels_originales"

    image_count = len(list(images_dir.glob("*")))
    label_count = len(list(labels_dir.glob("*.txt")))
    original_count = len(list(originals_dir.glob("*.txt")))

    print("\n" + "=" * 70)
    print(split)
    print("=" * 70)
    print(f"Imágenes:          {image_count}")
    print(f"Labels activas:    {label_count}")
    print(f"Labels originales: {original_count}")

    if image_count == 0:
        raise RuntimeError(f"No hay imágenes en {images_dir}")
    if original_count == 0:
        raise RuntimeError(f"No hay etiquetas originales en {originals_dir}")
    if label_count == 0:
        raise RuntimeError(f"No hay etiquetas activas en {labels_dir}")


ACCESSAI - ESTRUCTURA DEL DATASET

train
Imágenes:          19186
Labels activas:    19186
Labels originales: 19186

valid
Imágenes:          3511
Labels activas:    3511
Labels originales: 3511

test
Imágenes:          1629
Labels activas:    1629
Labels originales: 1629


## 08. Entrenamiento YOLO26x

In [20]:
# ============================================================
# ACCESSAI - 08. ENTRENAMIENTO OPTIMIZADO
# RTX 4060 Laptop GPU 8 GB
# YOLO26M - 4 CLASES
# ============================================================

import datetime
import gc
import sys
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO


print("=" * 80)
print("ACCESSAI - ENTRENAMIENTO YOLO26M OPTIMIZADO")
print("=" * 80)


# ============================================================
# 1. COMPROBAR CUDA
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible en este kernel de Jupyter.\n"
        "Selecciona el entorno 'proyecto_yolo' con PyTorch CUDA."
    )

device = 0

gpu_name = torch.cuda.get_device_name(device)

print(f"Python:       {sys.executable}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA PyTorch: {torch.version.cuda}")
print(f"GPU:          {gpu_name}")
print(f"GPU count:    {torch.cuda.device_count()}")


# ============================================================
# 2. COMPROBAR VRAM
# ============================================================

gc.collect()
torch.cuda.empty_cache()

# Reiniciar estadísticas de memoria
torch.cuda.reset_peak_memory_stats(device)

free_memory, total_memory = torch.cuda.mem_get_info(device)

free_gb = free_memory / 1024**3
total_gb = total_memory / 1024**3
used_gb = total_gb - free_gb

print("\n" + "-" * 80)
print("MEMORIA GPU")
print("-" * 80)

print(f"VRAM total:       {total_gb:.2f} GB")
print(f"VRAM usada:       {used_gb:.2f} GB")
print(f"VRAM disponible:  {free_gb:.2f} GB")

# No iniciar si prácticamente toda la GPU está ocupada
if free_gb < 2.0:
    raise RuntimeError(
        f"\n⚠️ Solo hay {free_gb:.2f} GB libres de VRAM.\n\n"
        "Reinicia el kernel de Jupyter y vuelve a ejecutar esta celda.\n"
        "También cierra otros procesos Python que puedan estar usando la GPU."
    )

print("✅ VRAM suficiente para iniciar el entrenamiento.")


# ============================================================
# 3. OPTIMIZACIONES CUDA / MATMUL
# ============================================================

torch.set_float32_matmul_precision("high")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("\nOptimizaciones CUDA:")
print("  TF32 matmul:      True")
print("  cuDNN TF32:       True")
print("  AMP:              True")
print("  Matmul precision: high")


# ============================================================
# 4. COMPROBAR DATASET
# ============================================================

if "yaml_nuevo_path" not in globals():
    raise RuntimeError(
        "No existe yaml_nuevo_path.\n"
        "Ejecuta primero las celdas de preparación del dataset."
    )

yaml_nuevo_path = Path(yaml_nuevo_path)

if not yaml_nuevo_path.exists():
    raise FileNotFoundError(
        f"No existe el YAML:\n{yaml_nuevo_path}"
    )

with open(
    yaml_nuevo_path,
    "r",
    encoding="utf-8"
) as f:
    data_yaml = yaml.safe_load(f)


dataset_nc = data_yaml.get("nc")
dataset_names = data_yaml.get("names")

if dataset_nc != 4:
    raise ValueError(
        f"El YAML tiene nc={dataset_nc}, "
        "pero este experimento necesita 4 clases."
    )

if not dataset_names or len(dataset_names) != 4:
    raise ValueError(
        f"El YAML contiene "
        f"{len(dataset_names) if dataset_names else 0} clases, "
        "pero deberían ser 4."
    )


print("\n" + "-" * 80)
print("DATASET")
print("-" * 80)

print("YAML:")
print(yaml_nuevo_path)

print("\nClases:")
for idx, name in enumerate(dataset_names):
    print(f"  {idx}: {name}")


# ============================================================
# 5. MODELO
# ============================================================

MODEL_NAME = "yolo26x.pt"

print("\n" + "-" * 80)
print("MODELO")
print("-" * 80)

print(f"Cargando: {MODEL_NAME}")

model = YOLO(MODEL_NAME)

print("✅ YOLO26M cargado correctamente")


# ============================================================
# 6. CONFIGURACIÓN OPTIMIZADA
# ============================================================

imgsz = 640

# AutoBatch:
# -1 => aproximadamente 60 % de la VRAM disponible
train_batch = -1

# En Windows/Jupyter 4 suele ser un punto razonable.
train_workers = 4

epochs = 30
patience = 10

optimizer = "AdamW"

lr0 = 0.001
lrf = 0.01

print("\n" + "-" * 80)
print("CONFIGURACIÓN")
print("-" * 80)

print(f"Modelo:          {MODEL_NAME}")
print(f"Imagen:          {imgsz}x{imgsz}")
print(f"Batch:           {train_batch}  (AutoBatch)")
print(f"Workers:         {train_workers}")
print(f"Epochs:          {epochs}")
print(f"Patience:        {patience}")
print(f"Optimizer:       {optimizer}")
print(f"lr0:             {lr0}")
print(f"lrf:             {lrf}")
print("AMP:             True")
print("Cache:           False")
print("Cos LR:          True")
print("Mosaic:          0.8")
print("Mixup:           0.0")
print("Close mosaic:    10")
print("Seed:            42")


# ============================================================
# 7. NOMBRE DEL EXPERIMENTO
# ============================================================

ahora_str = datetime.datetime.now().strftime(
    "%Y-%m-%d_%H-%M-%S"
)

runs_dir = Path(ROOT) / "runs"

experiment_name = (
    f"AccessAI_YOLO26M_4clases_{ahora_str}"
)

print("\n" + "-" * 80)
print("SALIDA")
print("-" * 80)

print(f"Project: {runs_dir}")
print(f"Name:    {experiment_name}")


# ============================================================
# 8. ENTRENAMIENTO
# ============================================================

print("\n" + "=" * 80)
print("🚀 INICIANDO ENTRENAMIENTO")
print("=" * 80)

results = model.train(

    # Dataset
    data=str(yaml_nuevo_path),

    # Modelo / resolución
    imgsz=imgsz,

    # AutoBatch
    batch=train_batch,

    # GPU
    device=device,

    # DataLoader
    workers=train_workers,

    # Entrenamiento
    epochs=epochs,
    patience=patience,

    # Optimizador
    optimizer=optimizer,
    lr0=lr0,
    lrf=lrf,

    # Scheduler
    cos_lr=True,

    # Precisión
    amp=True,

    # Augmentation
    mosaic=0.8,
    mixup=0.0,
    close_mosaic=10,

    # Reproducibilidad
    seed=42,
    deterministic=True,

    # Dataset
    cache=False,

    # Validación
    val=True,

    # Guardado
    save=True,
    save_period=-1,

    # Logs / gráficas
    plots=True,
    verbose=True,

    # Directorio
    project=str(runs_dir),
    name=experiment_name,
    exist_ok=False,
)


# ============================================================
# 9. RESULTADOS
# ============================================================

RUN_DIR = Path(results.save_dir)

BEST_MODEL = (
    RUN_DIR
    / "weights"
    / "best.pt"
)

LAST_MODEL = (
    RUN_DIR
    / "weights"
    / "last.pt"
)

RESULTS_CSV = RUN_DIR / "results.csv"


print("\n" + "=" * 80)
print("✅ ENTRENAMIENTO FINALIZADO")
print("=" * 80)

print("\nRun:")
print(RUN_DIR)

print("\nbest.pt:")
print(BEST_MODEL)
print("Existe:", BEST_MODEL.exists())

print("\nlast.pt:")
print(LAST_MODEL)
print("Existe:", LAST_MODEL.exists())

print("\nresults.csv:")
print(RESULTS_CSV)
print("Existe:", RESULTS_CSV.exists())


# ============================================================
# 10. MEMORIA GPU FINAL
# ============================================================

print("\n" + "-" * 80)
print("MEMORIA GPU FINAL")
print("-" * 80)

peak_reserved = (
    torch.cuda.max_memory_reserved(device)
    / 1024**3
)

peak_allocated = (
    torch.cuda.max_memory_allocated(device)
    / 1024**3
)

free_memory, total_memory = torch.cuda.mem_get_info(device)

print(f"Pico reservado:   {peak_reserved:.2f} GB")
print(f"Pico asignado:    {peak_allocated:.2f} GB")
print(f"VRAM libre final: {free_memory / 1024**3:.2f} GB")
print(f"VRAM total:       {total_memory / 1024**3:.2f} GB")


# ============================================================
# 11. RESUMEN AUTOMÁTICO
# ============================================================

print("\n" + "=" * 80)
print("RESUMEN")
print("=" * 80)

print(f"Modelo:      {MODEL_NAME}")
print(f"Resolución:  {imgsz}")
print(f"Batch:       AutoBatch (-1)")
print(f"Epochs:      {epochs}")
print(f"Workers:     {train_workers}")
print(f"GPU:         {gpu_name}")

print("\n✅ Experimento guardado en:")
print(RUN_DIR)

ACCESSAI - ENTRENAMIENTO YOLO26M OPTIMIZADO
Python:       c:\Users\Usuario\.conda\envs\proyecto_yolo\python.exe
PyTorch:      2.14.0+cu126
CUDA PyTorch: 12.6
GPU:          NVIDIA GeForce RTX 4060 Laptop GPU
GPU count:    1

--------------------------------------------------------------------------------
MEMORIA GPU
--------------------------------------------------------------------------------
VRAM total:       8.00 GB
VRAM usada:       1.07 GB
VRAM disponible:  6.92 GB
✅ VRAM suficiente para iniciar el entrenamiento.

Optimizaciones CUDA:
  TF32 matmul:      True
  cuDNN TF32:       True
  AMP:              True
  Matmul precision: high

--------------------------------------------------------------------------------
DATASET
--------------------------------------------------------------------------------
YAML:
C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset\data_filtrado.yaml

Clases:

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.


## 09. Cargar `best.pt`

In [ ]:
from ultralytics import YOLO

if not BEST_MODEL.exists():
    raise FileNotFoundError(f"No se encontró best.pt en:\n{BEST_MODEL}")

trained_model = YOLO(str(BEST_MODEL))

print("=" * 70)
print("ACCESSAI - MODELO ENTRENADO")
print("=" * 70)
print("Ruta:", BEST_MODEL)

print("\nClases:")
for idx, name in trained_model.names.items():
    print(f"{idx} -> {name}")

print("\n✅ Modelo AccessAI cargado")


## 10. Evaluación en `test`

In [ ]:
eval_batch = 16

test_results_dir = RUN_DIR / "test_eval"
test_results_dir.mkdir(parents=True, exist_ok=True)

metrics = trained_model.val(
    data=str(yaml_nuevo_path),
    split="test",
    imgsz=640,
    batch=eval_batch,
    device=device,
    plots=True,
    project=str(test_results_dir),
    name="results",
)

precision = float(metrics.box.mp)
recall = float(metrics.box.mr)

if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)
else:
    f1 = 0.0

print("\n" + "=" * 70)
print("MÉTRICAS TEST")
print("=" * 70)
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1:         {f1:.4f}")
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")

print("\nResultados de test guardados en:")
print(test_results_dir)


## 11. Curvas del entrenamiento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_CSV = RUN_DIR / "results.csv"

if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"No existe:\n{RESULTS_CSV}")

df = pd.read_csv(RESULTS_CSV)

print("Columnas disponibles:")
print(df.columns.tolist())


In [ ]:
# 11.1 Loss de entrenamiento
loss_columns = [
    "train/box_loss",
    "train/cls_loss",
    "train/dfl_loss",
]

available_losses = [c for c in loss_columns if c in df.columns]

plt.figure(figsize=(12, 6))
for column in available_losses:
    plt.plot(
        df["epoch"],
        df[column],
        label=column.replace("train/", "").replace("_", " ").title(),
    )

plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("AccessAI - Pérdidas de entrenamiento")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 11.2 mAP
map_columns = [
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]

available_map = [c for c in map_columns if c in df.columns]

plt.figure(figsize=(12, 6))
for column in available_map:
    plt.plot(
        df["epoch"],
        df[column],
        label=column.replace("metrics/", "").replace("(B)", ""),
    )

plt.xlabel("Época")
plt.ylabel("mAP")
plt.title("AccessAI - Evolución de mAP")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 11.3 Precision y Recall
pr_columns = [
    "metrics/precision(B)",
    "metrics/recall(B)",
]

available_pr = [c for c in pr_columns if c in df.columns]

plt.figure(figsize=(12, 6))
for column in available_pr:
    nombre = (
        column.replace("metrics/", "")
        .replace("(B)", "")
        .capitalize()
    )
    plt.plot(df["epoch"], df[column], label=nombre)

plt.xlabel("Época")
plt.ylabel("Valor")
plt.title("AccessAI - Precision y Recall")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 11.4 Loss de validación
validation_columns = [
    "val/box_loss",
    "val/cls_loss",
    "val/dfl_loss",
]

available_validation = [c for c in validation_columns if c in df.columns]

if available_validation:
    plt.figure(figsize=(12, 6))

    for column in available_validation:
        plt.plot(
            df["epoch"],
            df[column],
            label=column.replace("val/", "Val ").replace("_", " ").title(),
        )

    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title("AccessAI - Loss de validación")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No hay columnas de loss de validación en results.csv.")


In [ ]:
# 11.5 Resumen de métricas de la última época
metricas_finales = {}

column_map = {
    "mAP50": "metrics/mAP50(B)",
    "mAP50-95": "metrics/mAP50-95(B)",
    "Precision": "metrics/precision(B)",
    "Recall": "metrics/recall(B)",
}

for nombre, columna in column_map.items():
    if columna in df.columns:
        metricas_finales[nombre] = float(df[columna].iloc[-1])

if metricas_finales:
    plt.figure(figsize=(10, 6))
    nombres = list(metricas_finales.keys())
    valores = list(metricas_finales.values())

    plt.bar(nombres, valores)
    plt.ylabel("Valor")
    plt.ylim(0, 1)
    plt.title("AccessAI - Métricas de la última época")
    plt.grid(axis="y")

    for i, valor in enumerate(valores):
        plt.text(
            i,
            min(valor + 0.02, 0.98),
            f"{valor:.3f}",
            ha="center",
        )

    plt.tight_layout()
    plt.show()


## 12. Distribución de clases

In [ ]:
conteo_clases = {nombre: 0 for nombre in NUEVOS_NOMBRES}
train_labels_dir = DATASET_PATH / "train" / "labels"

if train_labels_dir.exists():
    for label_file in train_labels_dir.glob("*.txt"):
        with open(label_file, "r", encoding="utf-8") as f:
            for line in f:
                partes = line.strip().split()
                if not partes:
                    continue

                try:
                    clase_id = int(partes[0])
                except ValueError:
                    continue

                if 0 <= clase_id < NUM_CLASSES:
                    conteo_clases[NUEVOS_NOMBRES[clase_id]] += 1

    nombres = list(conteo_clases.keys())
    cantidades = list(conteo_clases.values())

    plt.figure(figsize=(12, 6))
    plt.bar(nombres, cantidades)
    plt.xlabel("Clase")
    plt.ylabel("Número de objetos")
    plt.title("AccessAI - Distribución de clases en train")
    plt.xticks(rotation=15)
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()

    print("\nDistribución de clases:")
    for nombre, cantidad in conteo_clases.items():
        print(f"{nombre:30s}: {cantidad}")
else:
    print("⚠️ No se encontró el directorio de labels de train.")


## 13. Matrices de confusión

In [ ]:
confusion_candidates = [
    RUN_DIR / "confusion_matrix.png",
    RUN_DIR / "confusion_matrix_normalized.png",
    test_results_dir / "results" / "confusion_matrix.png",
    test_results_dir / "results" / "confusion_matrix_normalized.png",
]

for confusion_path in confusion_candidates:
    if confusion_path.exists():
        image = plt.imread(confusion_path)

        plt.figure(figsize=(12, 10))
        plt.imshow(image)
        plt.axis("off")
        plt.title(f"AccessAI - {confusion_path.name}")
        plt.tight_layout()
        plt.show()

        print("Matriz encontrada:", confusion_path)


## 14. Seleccionar una imagen de prueba

In [ ]:
import cv2
import matplotlib.pyplot as plt

sample_candidates = [
    ROOT / "tests" / "imgs" / "image.png",
    ROOT / "tests" / "imgs" / "IMG_20170311_205902.jpg",
    DATASET_PATH / "test" / "images" / "IMG_19187.jpg",
]

image_path = next(
    (p for p in sample_candidates if p.exists()),
    None,
)

if image_path is None:
    # Fallback: primera imagen disponible del test.
    test_images = sorted((DATASET_PATH / "test" / "images").glob("*"))
    image_path = test_images[0] if test_images else None

if image_path is None:
    raise FileNotFoundError("No se encontró ninguna imagen de prueba.")

print("Imagen:")
print(image_path)


## 15. Inferencia

In [ ]:
results_predict = trained_model.predict(
    source=str(image_path),
    conf=0.25,
    imgsz=640,
    device=device,
    verbose=False,
)

result = results_predict[0]

print(f"Detecciones encontradas: {len(result.boxes)}")

if result.boxes is not None and len(result.boxes) > 0:
    for idx, box in enumerate(result.boxes, start=1):
        cls_id = int(box.cls[0])
        cls_name = trained_model.names[cls_id]
        confidence = float(box.conf[0])
        coords = [round(float(x), 2) for x in box.xyxy[0].tolist()]

        print(
            f"[{idx}] "
            f"{cls_name} | "
            f"conf={confidence:.3f} | "
            f"bbox={coords}"
        )
else:
    print("No se detectaron objetos con conf=0.25.")


## 16. Guardar y mostrar la imagen anotada

In [ ]:
output_dir = ROOT / "resultados"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "accessai_resultado.jpg"

annotated = result.plot()

ok = cv2.imwrite(str(output_path), annotated)

if not ok:
    raise IOError(f"No se pudo guardar la imagen en:\n{output_path}")

print("✅ Imagen guardada en:")
print(output_path)

annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 8))
plt.imshow(annotated_rgb)
plt.axis("off")
plt.title("AccessAI - Detección urbana")
plt.tight_layout()
plt.show()


## 17. Resumen del experimento

Conserva al menos:

- `best.pt`
- `last.pt`
- `results.csv`
- `confusion_matrix.png`
- `results.png`
- imagen anotada de inferencia
- mAP50
- mAP50-95
- Precision
- Recall
- F1 derivado de Precision y Recall

### Interpretación

Las métricas corresponden exclusivamente al experimento de cuatro categorías agrupadas sobre el ROD-Dataset. No deben presentarse como una medición directa de accesibilidad urbana.

El siguiente paso del proyecto sería estudiar un dataset/clase realmente específico de accesibilidad, por ejemplo para elementos como `sidewalk` o `curbramp`, si esas clases son necesarias para el objetivo final.